# Tutorial - Adding a New Runner

This notebook shows how to add a new backend runner to COFOLDER.

It focuses on two things:

1. What input COFOLDER provides to a runner.
2. What output a runner must write so the shared analytics pipeline can continue as normal.


## Overview

A COFOLDER runner is responsible for:

- checking whether the backend is installed
- loading backend-specific options
- performing backend-specific system preparation
- running one prediction repeat
- normalizing backend output into COFOLDER's canonical bundle

COFOLDER itself still handles:

- CLI parsing
- shared input handling
- repeat orchestration
- structure gathering
- analytics and result merging


In [ ]:
from dataclasses import fields

from cofolder.modules.runners.base import RunnerPreparation, RunnerRequest, RunnerResult


def describe_dataclass(cls):
    print(cls.__name__)
    print('-' * len(cls.__name__))
    for field in fields(cls):
        print(f"{field.name}: {field.type}")
    print()


describe_dataclass(RunnerPreparation)
describe_dataclass(RunnerRequest)
describe_dataclass(RunnerResult)


## Input Provided By COFOLDER

### `prepare_system(...)`

COFOLDER calls `prepare_system(...)` before writing the final system YAML used for execution.

The runner receives:

- `system_obj`: parsed system YAML object
- `options_obj`: runner-specific options object from `load_options(...)`
- `wrk_dir`: shared raw working directory
- `conformers`: optional conformer mode from the CLI
- `sdf_file`: optional SDF path from the CLI
- `logger`: workflow logger

The runner returns a `RunnerPreparation` object, which may contain:

- a modified `system_obj`
- a modified `options_obj`
- warnings to surface in the workflow log
- runtime context such as cache path or diffusion sample count

### `run(request)`

COFOLDER then calls `run(...)` once per repeat with a `RunnerRequest`.

Important fields include:

- `system_path`: the final YAML path written by COFOLDER
- `system_obj`: parsed system object for the run
- `repeat_dir`: backend-owned directory for a single repeat
- `seed`: per-repeat seed
- `timings`: optional timing collector
- `label_prefix`: per-repeat timing label


In [ ]:
required_repeat_tree = {
    'raw/repeat_<n>/normalized/': [
        'system_metrics.csv',
        'chain_metrics.csv',
        'manifest.json',
        'structures/',
    ]
}

required_repeat_tree


## Output Required For Further Analysis

Every runner must normalize its backend output into the same canonical bundle.

Required files inside `raw/repeat_<n>/normalized/`:

- `system_metrics.csv`
- `chain_metrics.csv`
- `manifest.json`
- `structures/` containing copied `.cif`, `.mmcif`, or `.pdb` files

COFOLDER later merges those files into:

- `results/system_metrics.csv`
- `results/chain_metrics.csv`
- `results/structures/`

### Minimum expected CSV columns

For `system_metrics.csv`:

- `cif_file`
- `model_name`
- `repeat`
- `diffusion_sample`

For `chain_metrics.csv`:

- `conf_chain_id`
- `cif_file`
- `model_name`
- `repeat`
- `diffusion_sample`

COFOLDER adds shared chain metadata later, such as `CHAIN_ID`, `ENTITY_TYPE`, and `ligand_molecule_id`.


In [ ]:
runner_skeleton = '''
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from cofolder.modules.runners.base import (
    BaseRunner,
    RunnerPreparation,
    RunnerRequest,
    RunnerResult,
)


class MyRunner(BaseRunner):
    name = "my-runner"
    capabilities = {"confidence_metrics"}

    def check_availability(self) -> tuple[bool, str | None]:
        return self.check_distribution_available(
            distribution_name="my-backend",
            missing_message=(
                "The selected 'my-runner' backend is not installed. "
                'Install it with `pip install "cofolder[my-runner]"`.'
            ),
        )

    def load_options(self, options_path: Path):
        return {"options_path": str(options_path)}

    def prepare_system(self, system_obj, options_obj, wrk_dir, conformers, sdf_file, logger):
        return RunnerPreparation(
            system_obj=system_obj,
            options_obj=options_obj,
            runtime_context={"diffusion_samples": 1},
        )

    def run(self, request: RunnerRequest) -> RunnerResult:
        raw_output_dir = request.repeat_dir / "my_backend_output"
        normalized_dir = request.repeat_dir / "normalized"
        structures_dir = normalized_dir / "structures"
        raw_output_dir.mkdir(parents=True, exist_ok=True)
        structures_dir.mkdir(parents=True, exist_ok=True)

        system_df = pd.DataFrame([
            {
                "cif_file": f"{request.repeat}_{request.system_name}_model_0.cif",
                "model_name": request.system_name,
                "repeat": request.repeat,
                "diffusion_sample": 0,
                "confidence_score": 0.91,
            }
        ])
        chain_df = pd.DataFrame([
            {
                "conf_chain_id": 0,
                "cif_file": f"{request.repeat}_{request.system_name}_model_0.cif",
                "model_name": request.system_name,
                "repeat": request.repeat,
                "diffusion_sample": 0,
            }
        ])

        system_metrics_path = normalized_dir / "system_metrics.csv"
        chain_metrics_path = normalized_dir / "chain_metrics.csv"
        system_df.to_csv(system_metrics_path, index=False)
        chain_df.to_csv(chain_metrics_path, index=False)

        sample_records = [
            {
                "repeat": request.repeat,
                "diffusion_sample": 0,
                "cif_file": f"{request.repeat}_{request.system_name}_model_0.cif",
            }
        ]

        manifest_path = normalized_dir / "manifest.json"
        manifest = {
            "runner": self.name,
            "capabilities": sorted(self.capabilities),
            "repeat": request.repeat,
            "raw_output_dir": str(raw_output_dir),
            "normalized_dir": str(normalized_dir),
            "system_metrics_path": str(system_metrics_path),
            "chain_metrics_path": str(chain_metrics_path),
            "structures_dir": str(structures_dir),
            "diffusion_samples": 1,
            "runtime_context": {"diffusion_samples": 1},
            "warnings": [],
            "sample_records": sample_records,
        }
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        return RunnerResult(
            runner_name=self.name,
            raw_output_dir=raw_output_dir,
            normalized_dir=normalized_dir,
            structures_dir=structures_dir,
            system_metrics_path=system_metrics_path,
            chain_metrics_path=chain_metrics_path,
            manifest_path=manifest_path,
            diffusion_samples=1,
            capabilities=set(self.capabilities),
            runtime_context={"diffusion_samples": 1},
            sample_records=sample_records,
        )


RUNNER = MyRunner()
'''

print(runner_skeleton)


In [ ]:
from cofolder.modules.runners import list_runner_names

list_runner_names()


## Capability Notes

A runner should declare only the metric groups it can really provide in normalized form.

Current shared metric groups are:

- `confidence_metrics`
- `affinity_metrics`
- `affinity_metrics_ext`

If a selected runner does not support a requested metric group, COFOLDER warns and continues. The output schema is still preserved by writing empty columns where needed.


## Practical Checklist

When adding a new runner:

1. Add a new module under `src/cofolder/modules/runners/`.
2. Expose a module-level `RUNNER` object.
3. Implement `check_availability()`, `load_options()`, `prepare_system()`, and `run()`.
4. Write the normalized bundle under `raw/repeat_<n>/normalized/`.
5. Add tests for runner discovery, availability, and normalized output generation.

If that contract is satisfied, COFOLDER's shared gathering and analytics pipeline can continue without backend-specific changes.
